In [ ]:

#!pip install -q --upgrade google-adk google-genai google-cloud-aiplatform pandas scikit-learn nest_asyncio
!pip install -q google-genai scikit-learn nest_asyncio

Import and API Key


In [ ]:
from google.colab import auth
import os

auth.authenticate_user()

PROJECT_ID = "crm-ai-agent-493618"
LOCATION = "us-central1"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

for key_name in ["GOOGLE_API_KEY", "GEMINI_API_KEY"]:
    os.environ.pop(key_name, None)


In [ ]:
#Import libraries
import json
import pandas as pd
import nest_asyncio

from google.colab import files
from google import genai
from google.genai import types

UPLOAD FILE

In [ ]:

uploaded = files.upload()


Saving Pham_Claire_cofi_ProjectProposalData.csv to Pham_Claire_cofi_ProjectProposalData.csv


In [ ]:
!pip install -q gradio #Install gradio
!gcloud services enable artifactregistry.googleapis.com \
  --project=crm-ai-agent-493618



To take a quick anonymous survey, run:
  $ gcloud survey



Clean Dataset

In [ ]:

# Change the filename if needed
CSV_FILE = "Pham_Claire_cofi_ProjectProposalData.csv"
lead_data = pd.read_csv(CSV_FILE)

# Remove junk column if present
if "Unnamed: 18" in lead_data.columns:
    lead_data = lead_data.drop(columns=["Unnamed: 18"])

# Basic cleanup
lead_data.columns = [c.strip() for c in lead_data.columns]

# Fill missing text fields
text_cols = [
    "first_name", "last_name", "full_name", "email", "phone", "company_name",
    "business_type", "job_title", "city", "state", "country", "timezone",
    "visit_channel", "lead_source_detail", "campaign_name",
    "preferred_contact_method", "opt_in_marketing", "lead_status",
    "preferred_bean_origin", "bean_preference", "roast_preference",
    "budget_range_usd", "recurring_order_interest", "downloaded_catalog",
    "requested_sample", "last_interaction_date", "next_best_action",
    "follow_up_priority", "assigned_sales_rep", "notes"
]
for col in text_cols:
    if col in lead_data.columns:
        lead_data[col] = lead_data[col].fillna("Unknown").astype(str)

# Fill missing numeric fields
num_cols = [
    "estimated_monthly_volume_kg", "website_pages_viewed",
    "time_on_site_seconds", "touchpoints_count", "urgency_days", "lead_score"
]
for col in num_cols:
    if col in lead_data.columns:
        lead_data[col] = pd.to_numeric(lead_data[col], errors="coerce").fillna(0)

print("Shape:", lead_data.shape)
print("\nColumns:")
print(list(lead_data.columns))
lead_data.head()

Shape: (150, 37)

Columns:
['lead_id', 'first_name', 'last_name', 'full_name', 'email', 'phone', 'company_name', 'business_type', 'job_title', 'city', 'state', 'country', 'timezone', 'visit_channel', 'lead_source_detail', 'campaign_name', 'preferred_contact_method', 'opt_in_marketing', 'lead_status', 'preferred_bean_origin', 'bean_preference', 'roast_preference', 'estimated_monthly_volume_kg', 'budget_range_usd', 'recurring_order_interest', 'website_pages_viewed', 'time_on_site_seconds', 'downloaded_catalog', 'requested_sample', 'touchpoints_count', 'urgency_days', 'last_interaction_date', 'lead_score', 'next_best_action', 'follow_up_priority', 'assigned_sales_rep', 'notes']


,lead_id,first_name,last_name,full_name,email,phone,company_name,business_type,job_title,city,...,downloaded_catalog,requested_sample,touchpoints_count,urgency_days,last_interaction_date,lead_score,next_best_action,follow_up_priority,assigned_sales_rep,notes
0,COFI-0020,Avery,Baker,Avery Baker,avery.baker@pacificcafe.co,-10759,Pacific Cafe,Hotel,Founder,New York,...,Yes,Yes,4,3,3/12/2026,98,Call within 1 hour + send pricing email,Urgent,E. Carter,Interested in recurring wholesale orders.
1,COFI-0016,Victoria,Rodriguez,Victoria Rodriguez,victoria.rodriguez@summittrading.coffee,-9572,Summit Trading,E-commerce Brand,Category Buyer,Boston,...,Yes,Yes,3,7,3/20/2026,98,Call within 1 hour + send pricing email,Urgent,D. Lopez,Requested pricing for private-label opportunit...
2,COFI-0049,Joseph,Moore,Joseph Moore,joseph.moore@rivercup.co,-5847,River Cup,Wholesale Distributor,Founder,Dallas,...,No,Yes,5,14,3/21/2026,98,Call within 1 hour + send pricing email,Urgent,B. Patel,Asked about shipping timelines for imported be...
3,COFI-0069,Amelia,Phillips,Amelia Phillips,amelia.phillips@summitmarket.coffee,-11161,Summit Market,Corporate Pantry Supplier,Owner,Nashville,...,Yes,Yes,3,14,3/23/2026,98,Call within 1 hour + send pricing email,Urgent,C. Nguyen,Exploring expansion to a second retail location.
4,COFI-0039,Gabriel,Martinez,Gabriel Martinez,gabriel.martinez@bluebean.coffee,-3614,Blue Bean,Office Distributor,Procurement Manager,Minneapolis,...,Yes,Yes,4,21,3/26/2026,98,Call within 1 hour + send pricing email,Urgent,D. Lopez,Exploring expansion to a second retail location.


In [ ]:
# =========================================
# Simple tool functions for Vertex AI workflow
# =========================================

def get_lead_by_id(lead_id: str) -> dict:
    """
    Retrieve one lead record from the CRM dataset by lead_id.
    Returns a plain dictionary for the workflow to analyze.
    """
    row = lead_data[lead_data["lead_id"].astype(str) == str(lead_id)]

    if row.empty:
        return {"error": f"Lead {lead_id} not found."}

    return row.iloc[0].to_dict()


def get_dataset_schema() -> dict:
    """
    Return the dataset schema so the AI workflow understands the available fields.
    """
    return {
        "columns": list(lead_data.columns),
        "row_count": int(len(lead_data))
    }


def segment_lead(lead_record: dict) -> dict:
    """
    Segment the lead into urgent, high, medium, or low.
    Uses follow_up_priority first.
    Falls back to lead_score if needed.
    """
    if "error" in lead_record:
        return lead_record

    priority = str(lead_record.get("follow_up_priority", "")).strip().lower()
    lead_score = lead_record.get("lead_score", None)

    valid_priorities = {"urgent", "high", "medium", "low"}

    if priority in valid_priorities:
        return {"lead_id": lead_record.get("lead_id"), "segment": priority}

    # Fallback using lead_score if follow_up_priority is missing
    try:
        score = float(lead_score)
        if score >= 80:
            segment = "urgent"
        elif score >= 60:
            segment = "high"
        elif score >= 40:
            segment = "medium"
        else:
            segment = "low"
    except (TypeError, ValueError):
        segment = "unknown"

    return {
        "lead_id": lead_record.get("lead_id"),
        "segment": segment
    }


def get_top_priority_leads(top_n: int = 10) -> dict:
    """
    Return the top leads using business fields in the dataset.
    Sorting preference:
    1) follow_up_priority (Urgent > High > Medium > Low)
    2) lead_score descending
    """
    tmp = lead_data.copy()

    priority_rank = {
        "urgent": 4,
        "high": 3,
        "medium": 2,
        "low": 1
    }

    tmp["priority_rank"] = (
        tmp["follow_up_priority"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(priority_rank)
        .fillna(0)
    )

    if "lead_score" in tmp.columns:
        tmp["lead_score"] = pd.to_numeric(tmp["lead_score"], errors="coerce").fillna(0)
        tmp = tmp.sort_values(["priority_rank", "lead_score"], ascending=[False, False])
    else:
        tmp = tmp.sort_values(["priority_rank"], ascending=[False])

    cols_to_keep = [
        c for c in [
            "lead_id",
            "full_name",
            "company_name",
            "lead_status",
            "lead_score",
            "follow_up_priority",
            "next_best_action",
            "preferred_contact_method",
            "assigned_sales_rep"
        ] if c in tmp.columns
    ]

    return {
        "top_leads": tmp.head(top_n)[cols_to_keep].to_dict(orient="records")
    }

In [ ]:
#VERTEX AI setup
from google import genai

client = genai.Client(
    vertexai=True,
    project="crm-ai-agent-493618",
    location="us-central1"
)
MODEL_NAME = "gemini-2.5-flash"
def ask_vertex_ai(prompt: str) -> str:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return response.text.strip()


WORKFLOW

In [ ]:
def build_crm_workflow():
    return {
        "name": "crm_workflow",
        "steps": [
            "data_summary",
            "qualification",
            "strategy",
            "outreach",
            "review",
            "sales_summary"
        ]
    }


Define Agents and Assign Tasks

In [ ]:
def run_crm_workflow(lead_id: str) -> dict:
    """
    Run the CRM workflow step by step using Vertex AI.
    """

    # Step 1: Retrieve lead data
    lead_record = get_lead_by_id(lead_id)
    if "error" in lead_record:
        return lead_record

    schema = get_dataset_schema()

    # Step 2: Data summary
    data_prompt = f"""
    You are the Lead Intake Agent for a coffee wholesale CRM.

    Your tasks:
    1. Review the lead record and dataset schema.
    2. Return a concise structured summary of the lead record using business language that easy to read.

    Lead ID:
    {lead_id}

    Dataset schema:
    {schema}

    Lead record:
    {lead_record}

    Rules:
    - Do not invent fields.
    - Keep the output factual and concise.
    """
    lead_record_summary = ask_vertex_ai(data_prompt)

    # Step 3: Qualification
    qualification_prompt = f"""
    You are the Lead Classifier Agent.

    Use this lead record:
    {lead_record}

    Evaluate the lead using the actual fields in the record, especially:
    - lead_score
    - lead_status
    - follow_up_priority
    - website_pages_viewed
    - time_on_site_seconds
    - requested_sample
    - recurring_order_interest
    - estimated_monthly_volume_kg
    - urgency_days
    - next_best_action
    - notes

    Your job:
    1. Decide how qualified the lead is.
    2. Explain the strongest buying-intent and engagement signals.
    3. Point out any weak or uncertain signals.
    4. Produce a structured output with:
      - qualification_label (Highly Qualified / Moderately Qualified / Low Qualified)
      - confidence_level (High / Medium / Low)
      - reasoning
      - key_signals
      - concerns

    Rules:
    - Base your answer only on the lead record.
    - Be business-oriented and concise.
    """
    qualification_result = ask_vertex_ai(qualification_prompt)

    # Step 4: Strategy
    strategy_prompt = f"""
    You are the Follow-up Strategy Agent.

    Use:
    Lead record: {lead_record}
    Qualification result: {qualification_result}

    Your job:
    1. Decide the best sales strategy for this lead.
    2. Choose the best channel based on preferred_contact_method and urgency.
    3. Align your recommendation with next_best_action when appropriate.
    4. Output:
      - recommended_strategy
      - recommended_channel
      - urgency_assessment
      - why_this_strategy

    Step to decide sales strategies:
    - High:
        Immediate personalized outreach (email, call, target marketing campaigns)
        Same-day follow-up
    - Medium:
        Recurring order discussion
        Offer new membership promotion
    - Low:
        Standard nurture follow-up
        Sales rep escalation

    Rules:
    - Use the lead's actual fields.
    - If dataset guidance and your judgment differ, mention that clearly.
    """
    strategy_result = ask_vertex_ai(strategy_prompt)

    # Step 5: Outreach
    outreach_prompt = f"""
    You are the Outreach Agent.

    Use:
    Lead record: {lead_record}
    Qualification result: {qualification_result}
    Strategy result: {strategy_result}

    Your job:
    1. Draft a personalized outreach message using persuasive tone.
    2. Adapt the tone to the lead's urgency, status, and preferred contact method.
    3. Use available profile details such as:
      - full_name
      - company_name
      - business_type
      - preferred_bean_origin
      - bean_preference
      - roast_preference
      - estimated_monthly_volume_kg
      - notes
    4. Output:
      - subject_line
      - outreach_channel
      - message_body
    5. Based on lead's urgency, offer promotion or discount accordingly.
       Put more effort into highly qualified leads.

    Rules:
    - Keep it professional and sales-ready.
    - Do not overclaim.
    - Make the message specific, not generic.
    - If preferred contact method is phone, write a short call script instead of an email.
    """
    outreach_result = ask_vertex_ai(outreach_prompt)

    # Step 6: Review
    review_prompt = f"""
    You are the Outreach Review Agent.

    Use:
    Lead record: {lead_record}
    Qualification result: {qualification_result}
    Strategy result: {strategy_result}
    Outreach draft: {outreach_result}

    Review the outreach for:
    - fit with qualification level
    - fit with urgency
    - fit with preferred contact method
    - clarity of call to action
    - professionalism
    - personalization

    Output:
    - approved (Yes or No)
    - review_feedback
    - revision_needed (Yes or No)
    - revised_message_body

    Rules:
    - If the draft is strong, keep revised_message_body mostly the same.
    - If weak, improve it directly and explain what is weak.
    - Be strict but practical.
    """
    review_result = ask_vertex_ai(review_prompt)

    # Step 7: Final sales summary
    sales_summary_prompt = f"""
    You are the Sales Summary Agent.

    Use:
    Lead record: {lead_record}
    Qualification result: {qualification_result}
    Strategy result: {strategy_result}
    Review result: {review_result}

    Write a clean final summary for a sales rep or manager.

    Output:
    - executive_summary
    - recommended_next_step
    - final_channel
    - final_message_to_use

    Rules:
    - final_message_to_use should use the revised message if the review changed it.
    - Keep the executive_summary concise and presentation-ready.
    """
    final_result = ask_vertex_ai(sales_summary_prompt)

    return {
        "lead_record": lead_record,
        "lead_record_summary": lead_record_summary,
        "qualification_result": qualification_result,
        "strategy_result": strategy_result,
        "outreach_result": outreach_result,
        "review_result": review_result,
        "final_result": final_result
    }


crm_workflow = build_crm_workflow()

In [ ]:
#Gradio UI setup with clean outputs
import gradio as gr

def format_title(text):
    return text.replace("_", " ").title()

def format_value(value):
    if value is None:
        return "N/A"
    if isinstance(value, bool):
        return "Yes" if value else "No"
    return str(value)

def gradio_crm_app(lead_id):
    if not lead_id:
        return "Please enter a Lead ID."

    try:
        result = run_crm_workflow(str(lead_id))

        if not isinstance(result, dict):
            return str(result)

        if "error" in result:
            return f"Error: {result['error']}"

        output = "# AI CRM Lead Analysis Result\n\n"

        # Lead record
        if "lead_record" in result:
            output += "## Lead Profile\n\n"
            lead = result["lead_record"]

            important_fields = [
                "lead_id", "full_name", "company_name", "business_type",
                "job_title", "city", "state", "lead_score",
                "lead_status", "follow_up_priority", "preferred_contact_method"
            ]

            for field in important_fields:
                if field in lead:
                    output += f"**{format_title(field)}:** {format_value(lead.get(field))}  \n"

            output += "\n---\n\n"

        # Lead summary
        if "lead_record_summary" in result:
            output += "## Lead Record Summary\n\n"
            output += f"{result['lead_record_summary']}\n\n"
            output += "---\n\n"

        # Qualification
        if "qualification_result" in result:
            output += "## Lead Qualification\n\n"
            output += f"{result['qualification_result']}\n\n"
            output += "---\n\n"

        # Strategy
        if "strategy_result" in result:
            output += "## Recommended Sales Strategy\n\n"
            output += f"{result['strategy_result']}\n\n"
            output += "---\n\n"

        # Outreach
        if "outreach_result" in result:
            output += "## Personalized Outreach Message\n\n"
            output += f"{result['outreach_result']}\n\n"
            output += "---\n\n"

        # Review
        if "review_result" in result:
            output += "## Outreach Review\n\n"
            output += f"{result['review_result']}\n\n"
            output += "---\n\n"

        # Final result
        if "final_result" in result:
            output += "## Final Sales Summary\n\n"
            output += f"{result['final_result']}\n\n"

        return output

    except Exception as e:
        return f"Error: {e}"

In [ ]:

demo = gr.Interface(
    fn=gradio_crm_app,
    inputs=gr.Textbox(label="Enter a lead ID:", placeholder="Example: COFI-1001"),
    outputs=gr.Markdown(label="CRM Agent Output"),
    title="AI CRM Lead Prioritization & Outreach",
    description="AI-powered CRM system that identifies high-potential leads and provides actionable insights for smarter sales decisions."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cc82761dd75f7d5bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print(run_crm_workflow("COFI-0020"))

{'lead_record': {'lead_id': 'COFI-0020', 'first_name': 'Avery', 'last_name': 'Baker', 'full_name': 'Avery Baker', 'email': 'avery.baker@pacificcafe.co', 'phone': '-10759', 'company_name': 'Pacific Cafe', 'business_type': 'Hotel', 'job_title': 'Founder', 'city': 'New York', 'state': 'NY', 'country': 'United States', 'timezone': 'America/New_York', 'visit_channel': 'Trade Show', 'lead_source_detail': 'Importer networking event', 'campaign_name': 'Spring Origin Launch', 'preferred_contact_method': 'Email', 'opt_in_marketing': 'Yes', 'lead_status': 'Contacted', 'preferred_bean_origin': 'Peru', 'bean_preference': 'Fair Trade', 'roast_preference': 'Flexible', 'estimated_monthly_volume_kg': 300, 'budget_range_usd': '$5,000-$12,500', 'recurring_order_interest': 'Yes', 'website_pages_viewed': 18, 'time_on_site_seconds': 0.0, 'downloaded_catalog': 'Yes', 'requested_sample': 'Yes', 'touchpoints_count': 4, 'urgency_days': 3, 'last_interaction_date': '3/12/2026', 'lead_score': 98, 'next_best_action